### ClimateWins Weather Prediction : Keras Layered Model (RNN/LSTM)

#### 1. Import Libraries and Data

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from numpy import unique
from numpy import reshape
from keras.models import Sequential
from keras.layers import Conv1D, Conv2D, Dense, BatchNormalization, Flatten, MaxPooling1D, Dropout
from keras.layers import LSTM
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Set display options to show all columns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
# Create path for faster data importation
# Create a path for faster data importation
path = r'C:\Users\harip\Desktop\Data Analytics Project\Machine Learning with Python\Exercise 2.2 - 2'
X = pd.read_csv(os.path.join(path, '02 Data', 'Prepared Data', 'weather_cleaned.csv'))
prediction = pd.read_csv(os.path.join(path, '02 Data', 'Original Data', 'Pleasant_Weather.csv'))
X.head()

,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,BASEL_temp_max,BELGRADE_cloud_cover,BELGRADE_humidity,BELGRADE_pressure,BELGRADE_global_radiation,BELGRADE_precipitation,BELGRADE_sunshine,BELGRADE_temp_mean,BELGRADE_temp_min,BELGRADE_temp_max,BUDAPEST_cloud_cover,BUDAPEST_humidity,BUDAPEST_pressure,BUDAPEST_global_radiation,BUDAPEST_precipitation,BUDAPEST_sunshine,BUDAPEST_temp_mean,BUDAPEST_temp_min,BUDAPEST_temp_max,DEBILT_cloud_cover,DEBILT_humidity,DEBILT_pressure,DEBILT_global_radiation,DEBILT_precipitation,DEBILT_sunshine,DEBILT_temp_mean,DEBILT_temp_min,DEBILT_temp_max,DUSSELDORF_cloud_cover,DUSSELDORF_humidity,DUSSELDORF_pressure,DUSSELDORF_global_radiation,DUSSELDORF_precipitation,DUSSELDORF_sunshine,DUSSELDORF_temp_mean,DUSSELDORF_temp_min,DUSSELDORF_temp_max,HEATHROW_cloud_cover,HEATHROW_humidity,HEATHROW_pressure,HEATHROW_global_radiation,HEATHROW_precipitation,HEATHROW_sunshine,HEATHROW_temp_mean,HEATHROW_temp_min,HEATHROW_temp_max,KASSEL_cloud_cover,KASSEL_humidity,KASSEL_pressure,KASSEL_global_radiation,KASSEL_precipitation,KASSEL_sunshine,KASSEL_temp_mean,KASSEL_temp_min,KASSEL_temp_max,LJUBLJANA_cloud_cover,LJUBLJANA_humidity,LJUBLJANA_pressure,LJUBLJANA_global_radiation,LJUBLJANA_precipitation,LJUBLJANA_sunshine,LJUBLJANA_temp_mean,LJUBLJANA_temp_min,LJUBLJANA_temp_max,MAASTRICHT_cloud_cover,MAASTRICHT_humidity,MAASTRICHT_pressure,MAASTRICHT_global_radiation,MAASTRICHT_precipitation,MAASTRICHT_sunshine,MAASTRICHT_temp_mean,MAASTRICHT_temp_min,MAASTRICHT_temp_max,MADRID_cloud_cover,MADRID_humidity,MADRID_pressure,MADRID_global_radiation,MADRID_precipitation,MADRID_sunshine,MADRID_temp_mean,MADRID_temp_min,MADRID_temp_max,MUNCHENB_cloud_cover,MUNCHENB_humidity,MUNCHENB_pressure,MUNCHENB_global_radiation,MUNCHENB_precipitation,MUNCHENB_sunshine,MUNCHENB_temp_mean,MUNCHENB_temp_min,MUNCHENB_temp_max,OSLO_cloud_cover,OSLO_humidity,OSLO_pressure,OSLO_global_radiation,OSLO_precipitation,OSLO_sunshine,OSLO_temp_mean,OSLO_temp_min,OSLO_temp_max,SONNBLICK_cloud_cover,SONNBLICK_humidity,SONNBLICK_pressure,SONNBLICK_global_radiation,SONNBLICK_precipitation,SONNBLICK_sunshine,SONNBLICK_temp_mean,SONNBLICK_temp_min,SONNBLICK_temp_max,STOCKHOLM_cloud_cover,STOCKHOLM_humidity,STOCKHOLM_pressure,STOCKHOLM_global_radiation,STOCKHOLM_precipitation,STOCKHOLM_sunshine,STOCKHOLM_temp_mean,STOCKHOLM_temp_min,STOCKHOLM_temp_max,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,7,0.85,1.018,0.32,0.09,0.7,6.5,0.8,10.9,1,0.81,1.0195,0.88,0.00,7.0,3.7,-0.9,7.9,4,0.67,1.017,0.44,0.01,2.3,2.4,-0.4,5.1,7,0.85,1.0032,0.07,0.25,0.0,9.3,7.4,11.0,8,0.83,1.0161,0.12,0.08,0.0,10.0,7.0,11.5,7,0.91,1.0010,0.13,0.22,0.0,10.6,9.4,8.3,8,0.82,1.0094,0.28,0.48,1.6,7.9,3.9,9.4,8,1.00,1.0173,0.20,0.00,0.0,-0.6,-1.9,0.5,7,0.83,1.0063,0.22,0.32,1.0,9.5,8.5,11.1,6,0.92,1.0260,0.53,0.0,1.4,7.6,4.4,10.8,5,0.67,1.018,0.20,0.10,0.0,6.9,1.1,10.4,8,0.98,0.9978,0.04,1.14,0.0,4.9,3.8,5.9,4,0.73,1.0304,0.48,0.01,2.3,-5.9,-8.5,-3.2,5,0.98,1.0114,0.05,0.32,0.0,4.2,2.2,4.9,5,0.88,1.0003,0.45,0.34,4.7,8.5,6.0,10.9
1,6,0.84,1.018,0.36,1.05,1.1,6.1,3.3,10.1,6,0.84,1.0172,0.25,0.00,0.0,2.9,2.2,4.4,4,0.67,1.017,0.18,0.31,0.0,2.3,1.4,3.1,8,0.90,1.0056,0.14,0.06,0.1,7.7,6.4,8.3,8,0.89,1.0161,0.18,0.66,0.5,8.2,7.4,11.0,7,0.98,1.0051,0.13,0.23,0.0,6.1,3.9,10.6,8,0.86,1.0086,0.12,0.27,0.0,7.7,6.8,9.1,6,0.94,1.0173,0.56,0.13,3.2,2.1,-1.3,5.5,8,0.92,1.0062,0.17,1.34,0.4,8.6,7.5,9.9,7,0.86,1.0254,0.46,0.0,0.9,9.8,7.4,12.2,6,0.72,1.018,0.61,0.30,5.1,6.2,4.2,10.2,8,0.62,1.0139,0.04,0.00,0.0,3.4,2.8,4.9,6,0.97,1.0292,0.21,0.61,0.0,-9.5,-10.5,-8.5,5,0.62,1.0114,0.05,0.06,0.0,4.0,3.0,5.0,7,0.91,1.0007,0.25,0.84,0.7,8.9,5.6,12.1
2,8,0.90,1.018,0.18,0.30,0.0,8.5,5.1,9.9,6,0.77,1.0179,0.67,0.00,3.5,3.1,-0.5,6.4,4,0.67,1.017,0.30,0.00,0.6,2.7,1.7,5.3,6,0.92,1.0165,0.28,0.01,

In [5]:
# check shape
X.shape

(22950, 135)

In [6]:
# check second dataset
prediction.head()

,DATE,BASEL_pleasant_weather,BELGRADE_pleasant_weather,BUDAPEST_pleasant_weather,DEBILT_pleasant_weather,DUSSELDORF_pleasant_weather,HEATHROW_pleasant_weather,KASSEL_pleasant_weather,LJUBLJANA_pleasant_weather,MAASTRICHT_pleasant_weather,MADRID_pleasant_weather,MUNCHENB_pleasant_weather,OSLO_pleasant_weather,SONNBLICK_pleasant_weather,STOCKHOLM_pleasant_weather,VALENTIA_pleasant_weather
0,19600101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,19600102,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,19600103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,19600104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,19600105,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
prediction.shape

(22950, 16)

#### 2. Data Wrangling and Reshaping

#### Data Wrangling

In [8]:
# Drop DATE column from answers

prediction.drop(columns = 'DATE', inplace = True)

In [9]:
prediction.shape

(22950, 15)

In [10]:
y = prediction

In [11]:
X.shape

(22950, 135)

In [12]:
# Turn X and y into arrays
X = np.array(X)
y = np.array(y)
X

array([[ 7.    ,  0.85  ,  1.018 , ...,  8.5   ,  6.    , 10.9   ],
       [ 6.    ,  0.84  ,  1.018 , ...,  8.9   ,  5.6   , 12.1   ],
       [ 8.    ,  0.9   ,  1.018 , ..., 10.5   ,  8.1   , 12.9   ],
       ...,
       [ 4.    ,  0.76  ,  1.0227, ..., 10.7   ,  7.9   , 13.5   ],
       [ 5.    ,  0.8   ,  1.0212, ..., 10.7   ,  7.9   , 13.5   ],
       [ 5.    ,  0.84  ,  1.0193, ..., 10.7   ,  7.9   , 13.5   ]])

In [13]:
X = X.reshape(-1,15,9)

In [14]:
# Verify Shape
X.shape

(22950, 15, 9)

In [15]:
# Verify Shape
y.shape

(22950, 15)

In [16]:
X

array([[[  7.    ,   0.85  ,   1.018 , ...,   6.5   ,   0.8   ,
          10.9   ],
        [  1.    ,   0.81  ,   1.0195, ...,   3.7   ,  -0.9   ,
           7.9   ],
        [  4.    ,   0.67  ,   1.017 , ...,   2.4   ,  -0.4   ,
           5.1   ],
        ...,
        [  4.    ,   0.73  ,   1.0304, ...,  -5.9   ,  -8.5   ,
          -3.2   ],
        [  5.    ,   0.98  ,   1.0114, ...,   4.2   ,   2.2   ,
           4.9   ],
        [  5.    ,   0.88  ,   1.0003, ...,   8.5   ,   6.    ,
          10.9   ]],

       [[  6.    ,   0.84  ,   1.018 , ...,   6.1   ,   3.3   ,
          10.1   ],
        [  6.    ,   0.84  ,   1.0172, ...,   2.9   ,   2.2   ,
           4.4   ],
        [  4.    ,   0.67  ,   1.017 , ...,   2.3   ,   1.4   ,
           3.1   ],
        ...,
        [  6.    ,   0.97  ,   1.0292, ...,  -9.5   , -10.5   ,
          -8.5   ],
        [  5.    ,   0.62  ,   1.0114, ...,   4.    ,   3.    ,
           5.    ],
        [  7.    ,   0.91  ,   1.0007, ...,   8.

#### 3. Data Splitting

In [18]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [19]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17212, 15, 9) (17212, 15)
(5738, 15, 9) (5738, 15)


#### 4. Keras Model Creation

In [20]:
epochs = 30
batch_size = 16
n_hidden = 64

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(MaxPooling1D())
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='tanh')) # Don't use relu here!

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 14, 64)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 7, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,215 (137.56 KB)

 Trainable params: 35,215 (137.56 KB)

 Non-trainable params: 0 (0.00 B)

#### 5. Compiling and Running Model

In [22]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [23]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/30
1076/1076 - 6s - 5ms/step - accuracy: 0.0840 - loss: 24.9389
Epoch 2/30
1076/1076 - 4s - 3ms/step - accuracy: 0.0578 - loss: 24.3794
Epoch 3/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0234 - loss: 24.6629
Epoch 4/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0669 - loss: 24.6696
Epoch 5/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0306 - loss: 24.6205
Epoch 6/30
1076/1076 - 4s - 4ms/step - accuracy: 0.0329 - loss: 24.4974
Epoch 7/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0400 - loss: 24.6745
Epoch 8/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0216 - loss: 24.3815
Epoch 9/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0504 - loss: 24.8428
Epoch 10/30
1076/1076 - 4s - 4ms/step - accuracy: 0.0154 - loss: 24.5083
Epoch 11/30
1076/1076 - 4s - 3ms/step - accuracy: 0.0155 - loss: 24.4291
Epoch 12/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0638 - loss: 24.5794
Epoch 13/30
1076/1076 - 3s - 3ms/step - accuracy: 0.1198 - loss: 24.7716
Epoch 14/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0733 - l

#### 6. Confusion Matrix Creation

In [25]:
# Define list of stations names

stations = {
0: 'BASEL',
1: 'BELGRADE',
2: 'BUDAPEST',
3: 'DEBILT',
4: 'DUSSELDORF',
5: 'HEATHROW',
6: 'KASSEL',
7: 'LJUBLJANA',
8: 'MAASTRICHT',
9: 'MADRID',
10: 'MUNCHENB',
11: 'OSLO',
12: 'SONNBLICK',
13: 'STOCKHOLM',
14: 'VALENTIA'

}

In [26]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [27]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step   
Pred        BELGRADE
True                
BASEL           3682
BELGRADE        1092
BUDAPEST         214
DEBILT            82
DUSSELDORF        29
HEATHROW          82
KASSEL            11
LJUBLJANA         61
MAASTRICHT         9
MADRID           458
MUNCHENB           8
OSLO               5
STOCKHOLM          4
VALENTIA           1


In [28]:
epochs = 30
batch_size = 16
n_hidden = 4

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax')) # Options: sigmoid, tanh, softmax, relu

In [29]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 14, 4)          │            76 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14, 16)         │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,851 (7.23 KB)

 Trainable params: 1,851 (7.23 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [31]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/30
1076/1076 - 3s - 3ms/step - accuracy: 0.0856 - loss: 403.5024
Epoch 2/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1078 - loss: 3844.6045
Epoch 3/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1214 - loss: 12314.1592
Epoch 4/30
1076/1076 - 3s - 2ms/step - accuracy: 0.1281 - loss: 27220.4746
Epoch 5/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1233 - loss: 48485.1680
Epoch 6/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1211 - loss: 77452.2969
Epoch 7/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1219 - loss: 112011.3125
Epoch 8/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1224 - loss: 155876.9062
Epoch 9/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1267 - loss: 210289.0625
Epoch 10/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1238 - loss: 271031.9688
Epoch 11/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1243 - loss: 347117.3125
Epoch 12/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1242 - loss: 428141.1875
Epoch 13/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1295 - loss: 525860.3125
Epoch 14/30
1076/

In [32]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [33]:
# Evaluate

print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Pred        BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  HEATHROW  KASSEL  MADRID  \
True                                                                           
BASEL              1       291     616          19       180       3     153   
BELGRADE           0       215     160           0        25       0      22   
BUDAPEST           0        50      28           0        15       0       6   
DEBILT             0         4      15           0         5       0       1   
DUSSELDORF         0         4      11           0         1       0       0   
HEATHROW           0         9      22           0        17       0      12   
KASSEL             0         3       3           0         0       0       0   
LJUBLJANA          0        18       1           0         2       0      12   
MAASTRICHT         0         0       2           0         1       0       0   
MADRID             0        71      56           0        37       0     142 

##### Softmax is not producing good results, trial with tanh, sigmoid, and relu

In [34]:
epochs = 30
batch_size = 16
n_hidden = 128

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='tanh')) # Options: sigmoid, tanh, softmax, relu

In [35]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 14, 128)        │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 14, 16)         │         2,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,191 (24.18 KB)

 Trainable params: 6,191 (24.18 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [37]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/30
1076/1076 - 3s - 3ms/step - accuracy: 0.1551 - loss: 24.0384
Epoch 2/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1756 - loss: 25.3739
Epoch 3/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1791 - loss: 23.8840
Epoch 4/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1782 - loss: 23.8241
Epoch 5/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1782 - loss: 23.8241
Epoch 6/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1782 - loss: 23.8278
Epoch 7/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1784 - loss: 23.8250
Epoch 8/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1786 - loss: 23.8269
Epoch 9/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1787 - loss: 23.8297
Epoch 10/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1788 - loss: 23.8362
Epoch 11/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1788 - loss: 23.8409
Epoch 12/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1798 - loss: 23.8466
Epoch 13/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1795 - loss: 23.8587
Epoch 14/30
1076/1076 - 2s - 2ms/step - accuracy: 0.1794 - l

In [43]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [44]:
# Evaluate

print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Pred        BELGRADE  DUSSELDORF  KASSEL  MADRID  MUNCHENB  OSLO  VALENTIA
True                                                                      
BASEL           1313         455      35     387      1382   109         1
BELGRADE         782         167       4     112        13    14         0
BUDAPEST         160          35       1      15         1     2         0
DEBILT            74           7       1       0         0     0         0
DUSSELDORF        23           2       1       3         0     0         0
HEATHROW          48          17       0      15         2     0         0
KASSEL             9           1       1       0         0     0         0
LJUBLJANA         23           6       0      25         6     1         0
MAASTRICHT         1           2       0       5         0     1         0
MADRID           112          82       3     126       127     8         0
MUNCHENB           2           3       0       2         0

##### Better loss but accuracy is low, lets try adjusting layers

In [59]:
epochs = 30
batch_size = 16
n_hidden = 64

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='tanh')) # Options: sigmoid, tanh, softmax, relu

In [60]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_5 (Conv1D)               │ (None, 14, 64)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 14, 16)         │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,951 (15.43 KB)

 Trainable params: 3,951 (15.43 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [62]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/30
1076/1076 - 4s - 4ms/step - accuracy: 0.1737 - loss: 27.8168
Epoch 2/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2224 - loss: 29.4652
Epoch 3/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2408 - loss: 29.0541
Epoch 4/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2446 - loss: 29.4381
Epoch 5/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2505 - loss: 29.5139
Epoch 6/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2440 - loss: 24.2380
Epoch 7/30
1076/1076 - 3s - 2ms/step - accuracy: 0.2487 - loss: 29.2901
Epoch 8/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2698 - loss: 29.1431
Epoch 9/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2776 - loss: 29.5551
Epoch 10/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2814 - loss: 29.5551
Epoch 11/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2873 - loss: 29.5551
Epoch 12/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2925 - loss: 29.5551
Epoch 13/30
1076/1076 - 2s - 2ms/step - accuracy: 0.2963 - loss: 29.5551
Epoch 14/30
1076/1076 - 3s - 2ms/step - accuracy: 0.2994 - l

In [63]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [64]:
# Evaluate

print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  
Pred        BASEL  BELGRADE  DEBILT  KASSEL  LJUBLJANA  MAASTRICHT
True                                                              
BASEL        3422       231       9       7          1          12
BELGRADE     1092         0       0       0          0           0
BUDAPEST      214         0       0       0          0           0
DEBILT         82         0       0       0          0           0
DUSSELDORF     29         0       0       0          0           0
HEATHROW       82         0       0       0          0           0
KASSEL         11         0       0       0          0           0
LJUBLJANA      61         0       0       0          0           0
MAASTRICHT      9         0       0       0          0           0
MADRID        458         0       0       0          0           0
MUNCHENB        8         0       0       0          0           0
OSLO            5         0       0       0          0           0
STOCKHOLM       4  

##### Loss is stable but higher than desired; accuracy improved with sigmoid and ReLU trials.

In [65]:
epochs = 30
batch_size = 16
n_hidden = 64

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='sigmoid')) # Options: sigmoid, tanh, softmax, relu

In [66]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 14, 64)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 14, 16)         │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,951 (15.43 KB)

 Trainable params: 3,951 (15.43 KB)

 Non-trainable params: 0 (0.00 B)

In [67]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [68]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/30
1076/1076 - 4s - 4ms/step - accuracy: 0.6195 - loss: 8351.7891
Epoch 2/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6433 - loss: 86109.5469
Epoch 3/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6433 - loss: 289676.0000
Epoch 4/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6437 - loss: 630319.8750
Epoch 5/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6438 - loss: 1149965.7500
Epoch 6/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6440 - loss: 1831081.1250
Epoch 7/30
1076/1076 - 3s - 2ms/step - accuracy: 0.6440 - loss: 2687730.5000
Epoch 8/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6440 - loss: 3707300.7500
Epoch 9/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6440 - loss: 4999250.5000
Epoch 10/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6440 - loss: 6526524.0000
Epoch 11/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6440 - loss: 8401259.0000
Epoch 12/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6440 - loss: 10422280.0000
Epoch 13/30
1076/1076 - 2s - 2ms/step - accuracy: 0.6440 - loss: 12900877.0000


In [69]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [70]:
# Evaluate

print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Pred        BASEL
True             
BASEL        3682
BELGRADE     1092
BUDAPEST      214
DEBILT         82
DUSSELDORF     29
HEATHROW       82
KASSEL         11
LJUBLJANA      61
MAASTRICHT      9
MADRID        458
MUNCHENB        8
OSLO            5
STOCKHOLM       4
VALENTIA        1
